In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
categories = pd.read_csv('product_categories.csv')
transactions = pd.read_csv('transactions.csv')
sessions = pd.read_csv('sessions.csv')

In [3]:
!pip install duckdb
import duckdb

   ---------------------------------------- 0.0/11.4 MB ? eta -:--:--
   -------------- ------------------------- 4.2/11.4 MB 22.9 MB/s eta 0:00:01
   ------------------------------------- -- 10.7/11.4 MB 28.0 MB/s eta 0:00:01
   ---------------------------------------- 11.4/11.4 MB 24.5 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
categories[categories['Item Category'].isna()]
categories['Item Category'] = categories['Item Category'].fillna("UNKNOWN")
categories['Item Sub-Category'] = categories['Item Category'].fillna("UNKNOWN")

In [20]:
transactions.head()

,machine_id,site_session_id,prod_category_id,prod_name,domain_id,prod_qty,prod_totprice,basket_tot,event_date,event_time,domain_name,total_transactions_2020,total_transactions_2021,total_transactions_2022,total_transactions_2023
0,285811167,7837313750729559995,1001001014,SaiLiiny Vest for Boys 3 Buttons Slim Fit Dres...,4.046670e+18,1.0,15.66,593.07,2020-01-01,0:01:03,amazon.com,69.0,NaN,NaN,NaN
1,285811167,7837313750729559995,1002004139,Mr and Mrs Cake Topper Rhinestone Crystal Meta...,4.046670e+18,1.0,17.99,593.07,2020-01-01,0:01:03,amazon.com,69.0,NaN,NaN,NaN
2,285811167,7837313750729559995,4005017229,"Hoffmaster 114000 Plastic Tablecover Roll, 300...",4.046670e+18,2.0,38.24,593.07,2020-01-01,0:01:03,amazon.com,69.0,NaN,NaN,NaN
3,285811167,7837313750729559995,1002004139,"HiiARug Handmade Wedding Bouquets, Artificial ...",4.046670e+18,1.0,35.98,593.07,2020-01-01,0:01:03,amazon.com,69.0,NaN,NaN,NaN
4,285811167,7837313750729559995,1001001014,CHAMA Men's Formal Classic Fit Business Dress ...,4.046670e+18,1.0,24.69,593.07,2020-01-01,0:01:03,amazon.com,69.0,NaN,NaN,NaN


In [5]:
transactions.site_session_id.astype(str)
sessions.site_session_id.astype(str)

0           6729633401426801739
1           1092751701181111355
2           7837313750729559995
3            192904185306363414
4           6798383743536217771
                   ...         
13765661    2640888765140691712
13765662     712868840266094016
13765663    5983590363902492002
13765664    4677707501469914285
13765665    4467295126475715506
Name: site_session_id, Length: 13765666, dtype: object

In [23]:
query = ''' 
SELECT *
FROM transactions
WHERE event_date  BETWEEN '2022-01-01' AND '2022-12-31'
''' 

filtered_transactions = duckdb.query(query).to_df()

In [24]:
filtered_transactions.site_session_id.nunique()

1

In [41]:
transactions.columns

Index(['machine_id', 'site_session_id', 'prod_category_id', 'prod_name',
       'domain_id', 'prod_qty', 'prod_totprice', 'basket_tot', 'event_date',
       'event_time', 'domain_name', 'total_transactions_2020',
       'total_transactions_2021', 'total_transactions_2022',
       'total_transactions_2023'],
      dtype='object')

In [42]:
sessions.columns

Index(['machine_id', 'site_session_id', 'user_session_id', 'domain_id',
       'ref_domain__name', 'pages_viewed', 'duration', 'event_date',
       'event_time', 'hoh_most_education', 'census_region', 'household_size',
       'hoh_oldest_age', 'household_income', 'children', 'racial_background',
       'connection_speed', 'hispanic', 'zip_code', 'domain_name'],
      dtype='object')

# EDA 

## Q1: Which demographic groups tend to spend more than expected based on their income, and how can we interpret or address these spending patterns?

Data needed: 

**Transactions**
- group by site session id and machine id -- find total basket and domain name
- machine_id 
- basket_tot 
- domain_name 

**Sessions**
- machine_id 
- site_session_id 
- demographics stuffs 


Join on the machine_id 

In [6]:
query = '''
SELECT machine_id, site_session_id, SUM(prod_qty) AS prod_count, AVG(basket_tot) as basket_tot, 
MAX(domain_name) AS domain_name, event_date
FROM transactions 
WHERE event_date BETWEEN '2022-01-01' AND '2022-12-31'
GROUP BY machine_id, site_session_id, event_date
ORDER BY event_date ASC, site_session_id DESC
'''
grouped_transactions = duckdb.query(query).to_df()

In [7]:
grouped_transactions

,machine_id,site_session_id,prod_count,basket_tot,domain_name,event_date
0,333061023,0,1.0,4.000,etsy.com,2022-01-01
1,350496301,0,14.0,129.390,target.com,2022-01-01
2,278498783,0,1.0,17.360,amazon.com,2022-01-01
3,248475212,0,6.0,107.760,target.com,2022-01-01
4,348626047,0,3.0,390.430,amazon.com,2022-01-01
...,...,...,...,...,...,...
172932,345024150,0,1.0,37.180,amazon.com,2022-12-31
172933,347392235,0,1.0,0.000,amazon.com,2022-12-31
172934,339363611,0,1.0,53.740,amazon.com,2022-12-31
172935,329875151,0,2.0,14.265,ebay.com,2022-12-31


In [8]:
query = ''' 
SELECT *
FROM sessions
WHERE event_date  BETWEEN '2022-01-01' AND '2022-12-31'
''' 

filtered_sessions = duckdb.query(query).to_df()

In [9]:
query = ''' 
SELECT machine_id, site_session_id, COUNT(site_session_id) AS session_count, event_date,
       event_time, hoh_most_education, census_region, household_size,
       hoh_oldest_age, household_income, children, racial_background,
       connection_speed, hispanic, zip_code, domain_name

FROM filtered_sessions 
GROUP BY machine_id, site_session_id, event_date,
       event_time, hoh_most_education, census_region, household_size,
       hoh_oldest_age, household_income, children, racial_background,
       connection_speed, hispanic, zip_code, domain_name
'''

grouped_sessions = duckdb.query(query).to_df()

In [10]:
grouped_sessions.head()

,machine_id,site_session_id,session_count,event_date,event_time,hoh_most_education,census_region,household_size,hoh_oldest_age,household_income,children,racial_background,connection_speed,hispanic,zip_code,domain_name
0,339298303,5930014918491235491,1,2022-02-10,5:07:26,-88.0,3.0,2.0,7.0,13.0,1.0,1.0,1.0,0.0,78223.0,amazon.com
1,341536421,5735297371689261268,1,2022-02-10,5:11:33,-88.0,3.0,NaN,2.0,18.0,1.0,1.0,1.0,0.0,32258.0,amazon.com
2,277124515,2340515527765005360,1,2022-02-10,5:18:15,-88.0,4.0,1.0,5.0,14.0,0.0,5.0,1.0,1.0,95207.0,ebay.com
3,337268924,5773359087478725953,1,2022-02-10,5:22:22,-88.0,2.0,5.0,9.0,11.0,1.0,1.0,1.0,0.0,65201.0,walmart.com
4,330666799,3597632980965022087,1,2022-02-10,5:39:47,-88.0,NaN,3.0,2.0,11.0,1.0,1.0,1.0,1.0,77047.0,amazon.com


In [25]:
# merging the two 
query = ''' 
SELECT * 
FROM grouped_transactions t 
INNER JOIN grouped_sessions s 
  ON t.machine_id = s.machine_id AND 
ORDER BY t.event_date 
'''
merged_df = duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

note: there's a problem with site_session_id for the transactions in 2022 (no unique ones, all 0) 

In [27]:
len(merged_df)

34849969

In [28]:
merged_df.drop(columns= ['site_session_id', 'event_date_1', 'domain_name_1', 'machine_id_1'], inplace=True)
merged_df.rename(columns={'site_session_id_1': 'site_session_id'}, inplace=True)
merged_df.head()

,machine_id,prod_count,basket_tot,domain_name,event_date,site_session_id,session_count,event_time,hoh_most_education,census_region,household_size,hoh_oldest_age,household_income,children,racial_background,connection_speed,hispanic,zip_code
0,249920695,1.0,38.050,ebay.com,2022-01-01,7929452121874482633,1,22:29:34,-88.0,4.0,3.0,10.0,12.0,1.0,3.0,NaN,0.0,92844.0
1,278931347,1.0,7.990,amazon.com,2022-01-01,2242111683432990202,1,23:44:49,-88.0,3.0,4.0,11.0,16.0,1.0,1.0,1.0,0.0,35242.0
2,356845575,2.0,10.300,amazon.com,2022-01-01,6993877606421951671,1,3:45:55,4.0,4.0,5.0,7.0,14.0,1.0,5.0,1.0,1.0,80920.0
3,315876607,4.0,33.210,ebay.com,2022-01-01,2271365079758904203,1,8:50:14,-88.0,2.0,5.0,9.0,16.0,1.0,1.0,1.0,0.0,53218.0
4,284084434,2.0,20.345,ebay.com,2022-01-01,3137879047785035591,1,0:14:55,-88.0,2.0,3.0,11.0,13.0,0.0,1.0,NaN,0.0,49525.0


In [29]:
merged_df.machine_id.nunique()

22463

In [30]:
merged_df.site_session_id.nunique()

1812542

In [31]:
merged_df.event_time.nunique()

86270

In [38]:
query = ''' 
SELECT machine_id, site_session_id, AVG(basket_tot) AS basket_tot, MAX(domain_name) AS domain_name,
        MAX(event_date) AS event_date, COUNT(*) AS session_count, MAX(event_time) AS event_time, 
        MAX(hoh_most_education) AS hoh_most_education, MAX(hoh_oldest_age) AS hoh_oldest_age,
        MAX(children) AS children, MAX(racial_background) AS racial_background,
        MAX(census_region) AS census_region, MAX(household_size) AS household_size, MAX(hispanic) AS hispanic, 
        MAX(zip_code) AS zip_code
FROM merged_df 
GROUP BY machine_id, site_session_id
'''
merged_grouped_df = duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [39]:
len(merged_grouped_df)

1812542

In [40]:
query = '''
SELECT machine_id, site_session_id, COUNT(DISTINCT event_date) AS event_date_count
FROM merged_df
GROUP BY machine_id, site_session_id
HAVING event_date_count > 1
'''
has_event_date_ge1 = duckdb.query(query).to_df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))